# Notebook 02 â€” Feature Engineering

Building the features I'll need for the statistical analysis in the next notebook: pace by race segment, in-race position changes, a fatigue proxy, and market-implied probability. One row per horse-race in the output.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

PROCESSED_DATA_DIR = Path("../data/processed")

race = pd.read_parquet(PROCESSED_DATA_DIR / "race.parquet")
start = pd.read_parquet(PROCESSED_DATA_DIR / "start.parquet")
tracking = pd.read_parquet(PROCESSED_DATA_DIR / "tracking.parquet")

print("race:", race.shape)
print("start:", start.shape)
print("tracking:", tracking.shape)

race: (2000, 10)
start: (14915, 9)
tracking: (5228430, 7)


Now that tracking is Parquet (~81MB vs. the original 340MB CSV) it's manageable to load fully into pandas. The grouped operations coming up (shift, interpolation) are easier to express there than in SQL.

## 1. Speed per frame and GPS noise cleanup

In [2]:
tracking = tracking.sort_values(
    ["track_id", "race_date", "race_number", "program_number", "trakus_index"]
)

group = tracking.groupby(["track_id", "race_date", "race_number", "program_number"])
tracking["prev_lat"] = group["latitude"].shift(1)
tracking["prev_lon"] = group["longitude"].shift(1)

In [3]:
def haversine_speed(lat1, lon1, lat2, lon2, dt=0.25):
    R = 6371000  # Earth radius in meters
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    distance_m = 2 * R * np.arcsin(np.sqrt(a))
    return distance_m / dt

tracking["speed_ms"] = haversine_speed(
    tracking["prev_lat"], tracking["prev_lon"], tracking["latitude"], tracking["longitude"]
)
tracking["speed_ms"].describe()

count    5.213515e+06
mean     1.845524e+01
std      2.848982e+00
min      0.000000e+00
25%      1.759086e+01
50%      1.901393e+01
75%      2.004202e+01
max      9.729080e+02
Name: speed_ms, dtype: float64

Matches what we found in notebook 01. Max speed in the hundreds of m/s, clearly GPS noise. Cleaning it now: flagging anything above 25 m/s (generous ceiling for a horse) as invalid, then filling those gaps by interpolating from the nearest valid frames for that same horse-race, rather than just capping the value at the threshold.

In [4]:
SPEED_THRESHOLD = 25  # m/s

n_noisy = (tracking["speed_ms"] > SPEED_THRESHOLD).sum()
print(f"Frames above threshold: {n_noisy}")

tracking.loc[tracking["speed_ms"] > SPEED_THRESHOLD, "speed_ms"] = np.nan

Frames above threshold: 1592


In [5]:
tracking["speed_ms"] = tracking.groupby(
    ["track_id", "race_date", "race_number", "program_number"]
)["speed_ms"].transform(lambda s: s.interpolate(limit_direction="both"))

tracking["speed_ms"].isnull().sum()

np.int64(0)

The NaN count right after flagging is higher than 1,592 because every horse-race's first frame also has no previous point to compare against, a normal side effect of the shift, not noise. Interpolation fills both.

In [6]:
tracking["speed_ms"].describe()

count    5.228430e+06
mean     1.840796e+01
std      2.760452e+00
min      0.000000e+00
25%      1.757158e+01
50%      1.900727e+01
75%      2.003813e+01
max      2.499835e+01
Name: speed_ms, dtype: float64

## 2. Race-segment pace features

Before building anything, testing whether tracking data actually joins with the other tables, since that's about to matter.

In [7]:
test_merge = tracking[["track_id", "race_date", "race_number", "program_number"]].drop_duplicates().merge(
    start[["track_id", "race_date", "race_number", "program_number", "odds"]],
    on=["track_id", "race_date", "race_number", "program_number"],
    how="left",
)

ValueError: You are trying to merge on object and datetime64[us] columns for key 'race_date'. If you wish to proceed you should use pd.concat

`race_date` in `tracking` came back from the Parquet export as a plain date object, not a datetime, so it doesn't match `start`. Casting it to datetime so the merge works.

In [8]:
tracking["race_date"] = pd.to_datetime(tracking["race_date"])

In [9]:
test_merge = tracking[["track_id", "race_date", "race_number", "program_number"]].drop_duplicates().merge(
    start[["track_id", "race_date", "race_number", "program_number", "odds"]],
    on=["track_id", "race_date", "race_number", "program_number"],
    how="left",
)
test_merge["odds"].isnull().sum()

np.int64(0)

0, fixed. Now excluding the one horse-race flagged in notebook 01: too few frames (4) to compute reliable segment features.

In [10]:
exclude = (
    (tracking["track_id"] == "BEL") & (tracking["race_date"] == "2019-07-06") &
    (tracking["race_number"] == 3) & (tracking["program_number"] == "3")
)
print("Rows excluded:", exclude.sum())
tracking = tracking[~exclude]

Rows excluded: 4


Splitting each horse's frames into three equal segments (early/mid/late) by relative progress through the race, then averaging speed within each. Using relative progress instead of raw `trakus_index`, since races have different lengths.

In [11]:
group = tracking.groupby(["track_id", "race_date", "race_number", "program_number"])
tracking["frame_rank"] = group.cumcount()
tracking["n_frames"] = group["trakus_index"].transform("count")
tracking["progress_pct"] = tracking["frame_rank"] / (tracking["n_frames"] - 1)

tracking["segment"] = pd.cut(
    tracking["progress_pct"], bins=[-0.01, 1 / 3, 2 / 3, 1.01], labels=["early", "mid", "late"]
)

In [12]:
segment_speed = (
    tracking.groupby(
        ["track_id", "race_date", "race_number", "program_number", "segment"], observed=True
    )["speed_ms"]
    .mean()
    .unstack("segment")
)
segment_speed.columns = [f"speed_{c}" for c in segment_speed.columns]
segment_speed = segment_speed.reset_index()
segment_speed.head()

,track_id,race_date,race_number,program_number,speed_early,speed_mid,speed_late
0,AQU,2019-01-01,1,1,18.753981,19.080092,16.886838
1,AQU,2019-01-01,1,2,18.175328,18.851025,17.080711
2,AQU,2019-01-01,1,3,18.663929,18.882340,16.401807
3,AQU,2019-01-01,1,4,18.495481,19.029263,16.574324
4,AQU,2019-01-01,1,5,18.397829,19.310504,17.536800


In [13]:
segment_speed[["speed_early", "speed_mid", "speed_late"]].mean()

speed_early    18.863191
speed_mid      19.294594
speed_late     17.221865
dtype: float64

Speed peaks mid-race (~19.3 m/s) and drops off noticeably in the late segment (~17.2 m/s). That late-race drop isn't just an average effect, 93% of horses show late speed below their own mid-race speed. The pattern is consistent across the dataset.

## 3. In-race position changes

Computing each horse's running position from cumulative distance traveled, then comparing early vs. late. Using each horse's own segment distances rather than matching `trakus_index` values across horses, not every horse in a race shares the exact same tracking range, so this avoids alignment issues.

In [14]:
tracking["distance_m"] = tracking["speed_ms"].fillna(0) * 0.25
tracking["cum_distance"] = tracking.groupby(
    ["track_id", "race_date", "race_number", "program_number"]
)["distance_m"].cumsum()

In [15]:
early_distance = (
    tracking[tracking["segment"] == "early"]
    .groupby(["track_id", "race_date", "race_number", "program_number"])["cum_distance"]
    .max()
    .rename("early_distance")
)

total_distance = (
    tracking.groupby(["track_id", "race_date", "race_number", "program_number"])["cum_distance"]
    .max()
    .rename("total_distance")
)

positions = pd.concat([early_distance, total_distance], axis=1).reset_index()

In [16]:
positions["early_position"] = positions.groupby(
    ["track_id", "race_date", "race_number"]
)["early_distance"].rank(ascending=False, method="min")

positions["late_position"] = positions.groupby(
    ["track_id", "race_date", "race_number"]
)["total_distance"].rank(ascending=False, method="min")

positions["position_change"] = positions["early_position"] - positions["late_position"]
positions.head()

,track_id,race_date,race_number,program_number,early_distance,total_distance,early_position,late_position,position_change
0,AQU,2019-01-01,1,1,496.980486,1441.112397,1.0,2.0,-1.0
1,AQU,2019-01-01,1,2,481.646198,1424.854267,5.0,3.0,2.0
2,AQU,2019-01-01,1,3,494.594118,1420.802972,2.0,5.0,-3.0
3,AQU,2019-01-01,1,4,490.130248,1424.724413,3.0,4.0,-1.0
4,AQU,2019-01-01,1,5,487.542465,1454.784210,4.0,1.0,3.0


`position_change` is positive when a horse moved up (better position late than early) and negative when it faded.

Sanity check: `late_position` here is derived purely from GPS distance, not from the official result. Comparing it against the real
`position_at_finish` from `start` tells us how good this proxy actually is.

In [17]:
check = positions.merge(
    start[["track_id", "race_date", "race_number", "program_number", "position_at_finish"]],
    on=["track_id", "race_date", "race_number", "program_number"],
    how="left",
)
print("Correlation:", check["late_position"].corr(check["position_at_finish"]))
print("Exact match rate:", (check["late_position"] == check["position_at_finish"]).mean())

Correlation: 0.8661530181113533
Exact match rate: 0.4824996647445353


Correlation of ~0.87 and an exact match rate of about 48%. Strong agreement, but not perfect, which makes sense: this is a GPS-distance proxy, not the actual photo-finish order, and the final stretch can be close.

## 4. Fatigue proxy

Fatigue proxy: the drop from mid-race speed (each horse's peak effort) to late-race speed. I'm using mid rather than early because the early segment includes the gate acceleration, which makes it noisy.

Standardizing within each race as a z-score, not across the full dataset. Race distance, track condition, and field quality all vary, what I want to know is whether a horse faded more than the others in its own race, not compared to a global baseline.

In [18]:
segment_speed["fatigue_raw"] = segment_speed["speed_mid"] - segment_speed["speed_late"]

race_group = segment_speed.groupby(["track_id", "race_date", "race_number"])["fatigue_raw"]
segment_speed["fatigue_z"] = race_group.transform(lambda s: (s - s.mean()) / s.std())

segment_speed["fatigue_z"].describe()

count    1.491400e+04
mean     1.259553e-17
std      9.305675e-01
min     -2.820007e+00
25%     -6.717848e-01
50%     -1.428783e-01
75%      6.290539e-01
max      3.399710e+00
Name: fatigue_z, dtype: float64

Validating this against something we already trust: if the fatigue signal means anything, horses with high fatigue_z should tend to finish worse.

In [19]:
check = segment_speed.merge(
    start[["track_id", "race_date", "race_number", "program_number", "position_at_finish"]],
    on=["track_id", "race_date", "race_number", "program_number"],
    how="left",
)
check["fatigue_z"].corr(check["position_at_finish"])

np.float64(0.48755123441430037)

Correlation of ~0.49. Moderate but clearly positive, confirming horses that fatigue more (relative to their own race) do tend to finish worse. Makes sense and gives confidence the signal is capturing something real, not noise.

In [20]:
(segment_speed["fatigue_z"] > 2).sum()

np.int64(320)

320 horses (about 2% of the dataset), show fatigue well above their race-mates (z > 2). These are the candidates for the early-warning framing: horses whose deceleration pattern stood out enough from the field that it might be worth a second look, not because something definitely went wrong, but because the pattern was unusual for that race.

## 5. Market feature â€” implied probability

Converting odds into implied win probability. Odds are stored as "X to 1" multiplied by 100 (e.g. 1280 = 12.8 to 1), decimal odds are X + 1, and implied probability is 1 / decimal_odds.

Excluding the 3 odds = 0 records flagged in notebook 01, these aren't real "even money" prices, they're missing values, and including them would wrongly imply a 100% win probability.

In [21]:
start["implied_prob"] = 1 / (start["odds"] / 100 + 1)
start.loc[start["odds"] == 0, "implied_prob"] = np.nan

start["implied_prob"].describe()

count    14912.000000
mean         0.163688
std          0.143915
min          0.005208
25%          0.054348
50%          0.120482
75%          0.229885
max          0.952381
Name: implied_prob, dtype: float64

Sanity check: grouping horses into deciles by implied probability and comparing against their actual win rate.

In [22]:
start["won"] = (start["position_at_finish"] == 1).astype(int)
start["prob_decile"] = pd.qcut(start["implied_prob"], 10, labels=False, duplicates="drop")

start.groupby("prob_decile").agg(
    avg_implied_prob=("implied_prob", "mean"),
    actual_win_rate=("won", "mean"),
    n=("won", "count"),
)

,avg_implied_prob,actual_win_rate,n
prob_decile,,,
0.0,0.017097,0.007338,1499
1.0,0.034076,0.026864,1489
2.0,0.054513,0.041916,1503
3.0,0.078848,0.059721,1507
4.0,0.105467,0.079863,1465
5.0,0.140365,0.112676,1562
6.0,0.181181,0.146207,1450
7.0,0.232540,0.196990,1462
8.0,0.309481,0.259452,1534


The actual win rate rises smoothly and consistently with implied probability across every decile. The market's relative ranking of horses tracks real outcomes well. But notice implied probability is consistently *higher* than the actual win rate in every single decile (e.g. the top decile: ~49% implied vs. ~42% actual wins). That's not a market error. Pari-mutuel odds bake in the track's commission ("takeout"), so implied probabilities calculated this way always run a bit above true odds across the board. Worth remembering: `implied_prob` is useful for ranking horses against each other, not as a literal probability.

## 6. Final feature table

In [23]:
features = segment_speed.merge(
    positions[["track_id", "race_date", "race_number", "program_number",
               "early_position", "late_position", "position_change"]],
    on=["track_id", "race_date", "race_number", "program_number"],
    how="left",
)

features = features.merge(
    start[["track_id", "race_date", "race_number", "program_number",
           "jockey", "weight_carried", "odds", "implied_prob", "position_at_finish"]],
    on=["track_id", "race_date", "race_number", "program_number"],
    how="left",
)

print(features.shape)
features.isnull().sum()

(14914, 17)


track_id              0
race_date             0
race_number           0
program_number        0
speed_early           0
speed_mid             0
speed_late            0
fatigue_raw           0
fatigue_z             0
early_position        0
late_position         0
position_change       0
jockey                0
weight_carried        0
odds                  0
implied_prob          3
position_at_finish    0
dtype: int64

14,914 rows, 17 columns â€” one row per horse-race. The only nulls are the 3 `implied_prob` values, the three records where `odds` was 0.

In [24]:
features.to_parquet(PROCESSED_DATA_DIR / "horse_race_features.parquet", index=False)